# Mock 5C · CarbonGlobe: 30-Year Forest Carbon Emulation · Mô phỏng carbon rừng toàn cầu trong 30 năm (Time Series · Chuỗi thời gian)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | The *Ecosystem Demography* model (ED v3) simulates how a forest grows from the weather it receives, but it is slow to run over the whole globe. Build a fast **emulator**: for a forest grid cell (0.5° ≈ 55 km) with a given initial stand age, you know the monthly weather of **1981–2020** and what ED simulated for **1980–1990**. Predict ED's 7 annual outputs for **every year 1991–2020** (a 30-year horizon). | Mô hình *Ecosystem Demography* (ED v3) mô phỏng rừng phát triển theo thời tiết, nhưng chạy cho cả địa cầu rất chậm. Hãy xây dựng một **mô hình thay thế (emulator)** nhanh: với một ô lưới rừng (0,5° ≈ 55 km) và tuổi rừng ban đầu cho trước, bạn biết thời tiết từng tháng giai đoạn **1981–2020** và kết quả ED mô phỏng cho **1980–1990**. Hãy dự đoán 7 đầu ra hằng năm của ED cho **mỗi năm 1991–2020** (tầm dự báo 30 năm). |
| **Data** | `train/train.npz`: 5,000 grid cells × 15 initial stand ages (1–500 years), with the **full** 1980–2020 trajectories. `public_test/` (1,993 samples) and `private_test/` (1,995 samples) are cells that are **not** in train (sample = cell + stand age). In both test sets about half of the cells come from the **hottest climates** (mean air temperature ≥ 26 °C), which train does not cover at all. | `train/train.npz`: 5.000 ô lưới × 15 tuổi rừng ban đầu (1–500 năm), có **đầy đủ** quỹ đạo 1980–2020. `public_test/` (1.993 mẫu) và `private_test/` (1.995 mẫu) là các ô **không** có trong train (mẫu = ô lưới + tuổi rừng). Trong cả hai tập test, khoảng một nửa số ô thuộc **vùng khí hậu nóng nhất** (nhiệt độ không khí trung bình ≥ 26 °C), hoàn toàn không có trong train. |
| **Metric** | NRMSE: RMSE of each variable divided by a fixed scale (its standard deviation in train), averaged over the 7 variables. **Lower is better.** Ranking uses the **private** score. | NRMSE: RMSE của từng biến chia cho một hằng số (độ lệch chuẩn của biến đó trong train), lấy trung bình trên 7 biến. **Càng thấp càng tốt.** Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv`: one row per sample and year, `id = <sample_id>_<year>` (1991–2020), columns `id,height,agb,soil_c,lai,gpp,npp,rh` | `public_submission.csv`, `private_submission.csv`: mỗi dòng là một mẫu và một năm, `id = <sample_id>_<năm>` (1991–2020), các cột `id,height,agb,soil_c,lai,gpp,npp,rh` |
| **Rules** | PyTorch model, **no external data**. The original CarbonGlobe dataset is public and contains the answers: downloading or using it in any way breaks the honour code. | Mô hình PyTorch, **không dùng dữ liệu ngoài**. Bộ dữ liệu CarbonGlobe gốc là công khai và chứa đáp án: tải về hoặc dùng nó dưới bất kỳ hình thức nào là vi phạm quy tắc danh dự. |

$$\text{NRMSE}=\frac{1}{7}\sum_{v=1}^{7}\frac{1}{s_v}\sqrt{\frac{1}{N}\sum_{i=1}^{N}\left(y_{i,v}-\hat y_{i,v}\right)^2}$$

🇬🇧 $N$ = rows (samples × 30 years), $s_v$ = the fixed scale of variable $v$ (`SCALE` below): predicting the training mean everywhere scores ≈ 1, keeping the 1990 value for 30 years ≈ 0.27–0.30.
🇻🇳 $N$ = số dòng (mẫu × 30 năm), $s_v$ = hằng số chuẩn hoá của biến $v$ (`SCALE` bên dưới): dự đoán giá trị trung bình của train cho mọi dòng được ≈ 1, giữ nguyên giá trị năm 1990 suốt 30 năm được ≈ 0,27–0,30.

**Files · Các file** (`np.load(path)`, every array has a name · mỗi mảng có tên):
- `train.npz`: `forcing` (cells, 480 months, 14), `static` (cells, 9), `targets` (cells, 15 ages, 41 years 1980–2020, 7) — **NaN** when ED simulated no forest for that cell and age · **NaN** khi ED không mô phỏng được rừng cho ô và tuổi đó; `ages`, `years`, `site_id`, `*_names`
- `public_test.npz`, `private_test.npz`: `sample_id`, `age` (initial stand age in 1980 · tuổi rừng năm 1980), `forcing` (n, 480, 14), `static` (n, 9), `history` (n, 11 years 1980–1990, 7), `history_years`, `target_years`
- Forcing (monthly, Jan 1981 – Dec 2020 · theo tháng): `t_air` (K), `precip`, `t_deep1..6` (temperatures that vary less and less, likely soil layers · nhiệt độ các lớp đất, càng sâu càng ít dao động), `co2` (ppm), `q_air` (humidity · độ ẩm), `t_range` (day–night range · biên độ ngày đêm, K), `sw_mean`, `sw_max` (sunlight · bức xạ mặt trời, W/m²), `wind` (m/s)
- Static · tĩnh: `soil_1..7` (soil parameters · thông số đất), `lat`, `lon` (rounded to 1° · làm tròn đến 1°)
- Targets · đầu ra (end of each year · cuối mỗi năm): `height` (tree height · chiều cao cây), `agb` (above-ground biomass · sinh khối trên mặt đất), `soil_c` (soil carbon · carbon trong đất), `lai` (leaf area index · chỉ số diện tích lá), `gpp`, `npp` (gross / net primary productivity · năng suất sơ cấp), `rh` (soil respiration · hô hấp dị dưỡng)

🇬🇧 A stand of initial age $a$ is $a + k$ years old in year $1980 + k$. Young stands grow fast, old ones are near equilibrium; some stands lose biomass after a bad year.
🇻🇳 Rừng có tuổi ban đầu $a$ sẽ $a + k$ tuổi vào năm $1980 + k$. Rừng non tăng trưởng nhanh, rừng già gần trạng thái cân bằng; có rừng bị mất sinh khối sau một năm xấu.

Data · Dữ liệu: derived from **CarbonGlobe** (Wang et al., *NeurIPS 2025 Datasets & Benchmarks*), [Kaggle](https://www.kaggle.com/datasets/zhihaow/carbonglobe) · [GitHub](https://github.com/zhwang0/carbon-globe), licence **CC BY 4.0** — see `dataset/README.md` for the changes made · xem các thay đổi trong `dataset/README.md`.
Related · Liên quan: [IEEE BigData Cup 2026 · AI Emulation Challenge (Kaggle)](https://www.kaggle.com/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge).

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "5C"
FOLDER  = "5C_carbon_globe"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/mock-5"

# Already have the folder? Point this at the parent of 5C_carbon_globe/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 5C_carbon_globe/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/5C_carbon_globe/dataset"
TRAIN_NPZ = f"{DATA_DIR}/train/train.npz"
PUBLIC_NPZ = f"{DATA_DIR}/public_test/public_test.npz"
PRIVATE_NPZ = f"{DATA_DIR}/private_test/private_test.npz"
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
VAL_FRACTION = 0.1  # share of train cells kept for validation / tỷ lệ ô train dùng để kiểm tra
BATCH_SIZE = 512
EPOCHS = 15
LR = 1e-3
HIDDEN = 256
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Features · Đọc dữ liệu và tạo đặc trưng

In [ ]:
train = dict(np.load(TRAIN_NPZ))
public = dict(np.load(PUBLIC_NPZ))
private = dict(np.load(PRIVATE_NPZ))
TARGETS = [str(t) for t in train["target_names"]]
FORCING_NAMES = [str(f) for f in train["forcing_names"]]
# EN: the metric's fixed scale of each variable (its SD in train, 1991-2020)
# VI: hằng số chuẩn hoá của từng biến trong độ đo (độ lệch chuẩn trong train, 1991-2020)
SCALE = np.array([10.3316, 5.3636, 6.2095, 2.0861, 1.4222, 0.6900, 0.6405], dtype=np.float32)
H = 11  # history years 1980-1990 / số năm lịch sử 1980-1990
print("train:", {k: v.shape for k, v in train.items() if v.ndim > 1})
print("public:", {k: v.shape for k, v in public.items() if v.ndim > 1})
print(f"Public samples: {len(public['sample_id'])} | Private samples: {len(private['sample_id'])}")
print("targets:", TARGETS)


# EN: standardisation statistics of the weather, from the training cells
# VI: thống kê chuẩn hoá thời tiết, lấy từ các ô train
F_MU = train["forcing"].reshape(-1, 14).mean(0)
F_SD = train["forcing"].reshape(-1, 14).std(0) + 1e-6


def weather_features(forcing):
    """EN: average weather of 1981-1990 and of 1991-2020 per variable (the monthly detail is thrown away).
    VI: thời tiết trung bình giai đoạn 1981-1990 và 1991-2020 của từng biến (bỏ qua chi tiết từng tháng)."""
    f = (forcing - F_MU) / F_SD
    return np.concatenate([f[:, :120].mean(1), f[:, 120:].mean(1)], 1)  # first 10 years = 120 months


def static_features(s):
    s = s.copy()
    s[:, 0] = np.log10(s[:, 0])  # soil_1 spans 5,000 .. 4,000,000 / soil_1 trải từ 5.000 đến 4.000.000
    return s


TRAIN_WEATHER = weather_features(train["forcing"])  # one row per train cell / mỗi ô train một dòng


def train_samples(cells):
    """EN: turn (cell, age) pairs of train into test-like samples (skip ages with no forest = NaN).
    VI: biến các cặp (ô, tuổi) của train thành mẫu giống tập test (bỏ các tuổi không có rừng = NaN)."""
    c, a = np.nonzero(~np.isnan(train["targets"][cells, :, 0, 0]))
    y = train["targets"][cells[c], a]
    return {"weather": TRAIN_WEATHER[cells[c]], "static": train["static"][cells[c]],
            "age": train["ages"][a], "history": y[:, :H], "future": y[:, H:]}


# EN: split by cell, so that validation cells are unseen like the test cells
# VI: tách theo ô lưới để các ô validation chưa từng thấy, giống tập test
rng = np.random.default_rng(SEED)
cells = rng.permutation(len(train["site_id"]))
n_val = int(VAL_FRACTION * len(cells))
tr_s, va_s = train_samples(cells[n_val:]), train_samples(cells[:n_val])
for d in (public, private):
    d["weather"] = weather_features(d["forcing"])
S_MU, S_SD = static_features(tr_s["static"]).mean(0), static_features(tr_s["static"]).std(0) + 1e-6


def make_features(d):
    """EN: one flat vector per sample: the 1980-1990 history (scaled), the stand age, static and weather features.
    VI: mỗi mẫu là một vector: lịch sử 1980-1990 (đã chuẩn hoá), tuổi rừng, đặc trưng tĩnh và thời tiết."""
    x = np.concatenate([(d["history"] / SCALE).reshape(len(d["age"]), -1), d["age"][:, None] / 500.0,
                        (static_features(d["static"]) - S_MU) / S_SD, d["weather"]], 1)
    return torch.tensor(x, dtype=torch.float32)


def make_target(d):
    """EN: change from the 1990 value, scaled: (y_year - y_1990) / SCALE. VI: mức thay đổi so với năm 1990, đã chuẩn hoá."""
    return torch.tensor(((d["future"] - d["history"][:, -1:]) / SCALE).reshape(len(d["age"]), -1), dtype=torch.float32)


X_train, y_train = make_features(tr_s), make_target(tr_s)
X_val = make_features(va_s)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
print(f"Training samples: {len(X_train):,} | Validation samples: {len(X_val):,} | features: {X_train.shape[1]}")

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
class CarbonMLP(nn.Module):
    """EN: flat features -> MLP -> 30 years x 7 variables (scaled change from 1990).
    VI: vector đặc trưng -> MLP -> 30 năm x 7 biến (mức thay đổi so với 1990, đã chuẩn hoá)."""

    def __init__(self, n_in, hidden=HIDDEN, n_out=30 * 7):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_out))

    def forward(self, x):
        return self.net(x)


model = CarbonMLP(X_train.shape[1]).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
def nrmse(y_true, y_pred):
    """EN: the competition metric on arrays (n, 30, 7). VI: độ đo của cuộc thi trên mảng (n, 30, 7)."""
    rmse = np.sqrt(((y_true - y_pred) ** 2).reshape(-1, 7).mean(0))
    return float((rmse / SCALE).mean())


@torch.no_grad()
def predict_future(model, d):
    """EN: forecasts in the original units, shape (n, 30, 7). VI: dự báo theo đơn vị gốc, kích thước (n, 30, 7)."""
    model.eval()
    out = model(make_features(d).to(DEVICE)).cpu().numpy().reshape(-1, 30, 7)
    return d["history"][:, -1:] + out * SCALE


persist = np.repeat(va_s["history"][:, -1:], 30, axis=1)
print(f"Validation NRMSE of 'keep the 1990 value' / giữ nguyên giá trị 1990: {nrmse(va_s['future'], persist):.4f}")

criterion = nn.MSELoss()  # on the scaled change = the metric's weighting / trên giá trị đã chuẩn hoá, khớp với độ đo
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val = float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)

    val = nrmse(va_s["future"], predict_future(model, va_s))
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(X_train):.4f} | Val NRMSE={val:.4f}"
    if val < best_val:
        best_val = val
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation NRMSE: {best_val:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
YEARS = np.arange(1991, 2021)


def export(d, preds, csv_path):
    """EN: one row per (sample, year): id = <sample_id>_<year>. Save the CSV and zip it.
    VI: mỗi dòng là một (mẫu, năm): id = <sample_id>_<năm>. Lưu CSV và nén zip."""
    df = pd.DataFrame(preds.reshape(-1, 7), columns=TARGETS)
    df.insert(0, "id", [f"{s}_{y}" for s in d["sample_id"] for y in YEARS])
    df.to_csv(csv_path, index=False, float_format="%.6g")
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} ({len(df):,} rows) and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public, predict_future(model, public), "public_submission.csv")
export(private, predict_future(model, private), "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def check_submission(csv_path, d):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == ["id", *TARGETS], f"{csv_path}: columns must be {['id', *TARGETS]}"
    want = {f"{s}_{y}" for s in d["sample_id"] for y in range(1991, 2021)}
    assert len(sub) == len(want) and set(sub["id"]) == want, f"{csv_path}: ids do not match the test set"
    assert sub[TARGETS].notna().all().all(), f"{csv_path}: contains missing values / có giá trị bị thiếu"
    assert np.isfinite(sub[TARGETS].to_numpy()).all(), f"{csv_path}: values must be finite / giá trị phải hữu hạn"


check_submission("public_submission.csv", public)
check_submission("private_submission.csv", private)

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "5C"          # which problem this notebook is for
CONTEST = "mock-5"    # this notebook is for Mock Contest 5
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")